# 鉄道写真オートトリミング モデル学習ノートブック（Keras → TFLite 用）

教師データ（画像 + 同名 `.jpg.json`）から、次の **2つの Keras モデル** を学習します。

| モデル | 入力 | 出力 |
|---|---|---|
| `trim_analyzer`（解析モデル） | 元画像（レターボックスで正方形化） | `rotation`: 回転角（度, 時計回りが正）／`aspect`: 5クラスの縦横比確率／`portrait`: 縦構図確率 |
| `trim_cropper`（トリミングモデル） | 回転後キャンバス画像 + 指定した縦横比の条件ベクトル | `box`: `[left, top, width, height]`（キャンバスに対する 0〜1 の比率。指定比率を厳密に満たす） |

推論の流れ: 解析モデルで角度と比率を推定 → 画像を回転 → トリミングモデルで BBox を算出（比率はユーザー指定でも可）。

## 教師データの仕様（実データで検証済み）
- `edit.rotation_degrees`: **正の値 = 時計回り** に回転させると水平になる角度。
- `edit.crop`: 回転後の画像全体が収まるよう **拡張したキャンバス**（幅 `W·cosθ + H·sinθ`、高さ `W·sinθ + H·cosθ`）に対する比率。
- `aspect_width` / `aspect_height`: `portrait: true` のとき縦横が入れ替わる（例: 3:2 縦 → 幅:高さ = 2:3）。
- `aspect_width: null` は「元画像の比率のまま」（データ上はすべて 4:3）。crop の実ピクセル比から最も近いクラスに割り当てます。
- `quarter_turns`（90度回転）は JPEG に既に適用済みのため無視します。
- 空の JSON や画像が無いものは自動でスキップします。

## 使い方（Google Colab）
1. 教師データフォルダを **zip にして** Google Drive にアップロード（フォルダのままでも可ですが、6,000ファイルのコピーは遅いです）。
2. メニュー「ランタイム → ランタイムのタイプを変更」で **GPU（T4 など）** を選択。
3. 下の「設定」セルの `DATA_SOURCE` と `OUTPUT_DIR` を自分の Drive のパスに合わせて、上から順に実行。
4. モデルは `OUTPUT_DIR/日時/` に保存されます。

## バックボーンの切り替え
「設定」セルの `A_BACKBONE`（解析モデル）と `B_BACKBONE`（トリミングモデル）に名前を書くだけで切り替わります。選べる名前は共通関数セルの `BACKBONES` を参照してください。

## データを追加したとき
- zip を差し替えて上から実行すれば、新しい zip を検出して展開し直します。
- 学習/検証の分割は撮影シーン名のハッシュで決まるため、既存の画像の割り当ては変わりません。
- 画像が増えてメモリに収まらなくなると（`MAX_MEMORY_CACHE_GB` 超え）、自動でディスクキャッシュに切り替わります。
- 一から学習し直す代わりに、`INIT_FROM_DIR` に前回の保存フォルダを指定すると、その重みから追加学習します（バックボーンと入力サイズが同じ場合のみ）。
- 評価セルの最後に、これまでの学習結果（データ数・回転角の誤差・比率の正解率・IoU）の一覧が出ます。

In [ ]:
import os, sys, datetime
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

## 設定

In [ ]:
# 教師データの場所（zip でもフォルダでも可）
DATA_SOURCE = os.environ.get('AUTOTRIM_DATA', '/content/drive/MyDrive/autotrimming/オートトリミングのための教師データ.zip')
# 学習済みモデルの保存先
OUTPUT_DIR = os.environ.get('AUTOTRIM_OUTPUT', '/content/drive/MyDrive/autotrimming/models')
LOCAL_DATA_DIR = '/content/data'   # Colab のローカルディスクにコピーして高速化

# 動作確認モード（少量データ・1エポックで最後まで流すだけ）
QUICK_TEST = os.environ.get('AUTOTRIM_QUICK_TEST') == '1'

SEED = 42
VAL_RATIO = 0.15          # 撮影シーン単位で固定的に割り当てるので、データが増えても既存の検証データは検証のまま
CACHE_LONG_SIDE = 512     # 学習に使う画像の長辺(px)。これより大きい画像は縮小してキャッシュ
CACHE_IN_MEMORY = 'auto'  # True: 全画像をメモリに載せる / False: 縮小画像をディスクに保存して都度読む / 'auto': 推定サイズで自動判定
MAX_MEMORY_CACHE_GB = 6.0 # 'auto' のときにメモリに載せる上限
DISK_CACHE_DIR = '/content/cache' if IN_COLAB else os.path.join(os.path.expanduser('~'), '.cache', 'autotrim')

# 保存先のサブフォルダ名（学習ごとに別フォルダにして、データを増やした前後のモデルを比較できるようにする）
RUN_NAME = datetime.datetime.now().strftime('%Y%m%d_%H%M')
# 前回の学習結果フォルダ（例: OUTPUT_DIR + '/20261110_1530'）。指定するとその重みから追加学習する。バックボーンと入力サイズは同じにすること
INIT_FROM_DIR = None

# バックボーン（下の BACKBONES の名前から選ぶ。解析モデルとトリミングモデルで別々に指定可）
#   軽い順: MobileNetV3Small < MobileNetV3Large < EfficientNetB0 / EfficientNetV2B0 < EfficientNetV2B1 < EfficientNetB2 / EfficientNetV2B2 < EfficientNetB3 / EfficientNetV2B3
#   ほかに MobileNetV2, ResNet50, ConvNeXtTiny も可（スマホ向けには重め）
A_BACKBONE = 'MobileNetV3Large'
B_BACKBONE = 'MobileNetV3Large'

# 解析モデル（回転角・比率）
A_SIZE = 384
A_BATCH = 32
A_EPOCHS_HEAD = 8         # バックボーン凍結で学習
A_EPOCHS_FINE = 40        # 全体を微調整
ROT_AUG_PROB = 0.7        # 回転の水増し確率
ROT_AUG_MAX = 10.0        # 水増しで加える最大角度(度)

# トリミングモデル（BBox）
B_SIZE = 320
B_BATCH = 32
B_EPOCHS_HEAD = 8
B_EPOCHS_FINE = 40
CROP_AUG_PROB = 0.3       # BBoxを含む範囲でキャンバスをランダムに切り出す水増し確率

if QUICK_TEST:
    A_EPOCHS_HEAD = A_EPOCHS_FINE = B_EPOCHS_HEAD = B_EPOCHS_FINE = 1
DATA_WORKERS = 4          # 学習中に画像を読み込むスレッド数（ディスクキャッシュ時に遅ければ増やす）

RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
os.makedirs(RUN_DIR, exist_ok=True)
import json
with open(os.path.join(RUN_DIR, 'config.json'), 'w') as f:
    json.dump({k: v for k, v in globals().items()
               if k.isupper() and isinstance(v, (int, float, str, bool, type(None)))}, f, indent=2, ensure_ascii=False)
print('保存先:', RUN_DIR)

## データの展開

In [ ]:
import shutil, zipfile, glob

def prepare_data(src, dst):
    if not IN_COLAB and os.path.isdir(src):
        return src                      # ローカル実行時はそのまま使う
    if src.lower().endswith('.zip'):
        st = os.stat(src)
        stamp = f'{st.st_size}-{int(st.st_mtime)}'
        marker = os.path.join(dst, '.source_stamp')
        if not (os.path.exists(marker) and open(marker).read() == stamp):
            shutil.rmtree(dst, ignore_errors=True)       # zip が更新されていたら展開し直す
            os.makedirs(dst, exist_ok=True)
            with zipfile.ZipFile(src) as z:
                for m in z.infolist():
                    if m.filename.startswith('__MACOSX') or '/._' in m.filename:
                        continue
                    z.extract(m, dst)
            with open(marker, 'w') as f:
                f.write(stamp)
    else:
        shutil.copytree(src, dst, dirs_exist_ok=True)
    jsons = glob.glob(os.path.join(dst, '**', '*.jpg.json'), recursive=True)
    return os.path.dirname(jsons[0])

DATA_DIR = prepare_data(DATA_SOURCE, LOCAL_DATA_DIR)
print('DATA_DIR =', DATA_DIR)

## ライブラリと共通関数（幾何変換）

In [ ]:
import json, re, math, random, unicodedata
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers, ops

print('TF', tf.__version__, '/ Keras', keras.__version__, '/ GPU', tf.config.list_physical_devices('GPU'))
random.seed(SEED); np.random.seed(SEED); keras.utils.set_random_seed(SEED)

ASPECTS = [(1, 1), (5, 4), (4, 3), (3, 2), (16, 9)]       # (aspect_width, aspect_height)
ASPECT_NAMES = ['1:1', '5:4', '4:3', '3:2', '16:9']
NUM_ASPECTS = len(ASPECTS)
COND_DIM = NUM_ASPECTS + 2                                 # one-hot + portrait + log(正規化比率)

# 名前 -> (クラス, 追加引数, 前処理)。前処理が None のものはモデル内で 0〜255 を正規化する
_app = keras.applications
BACKBONES = {
    'MobileNetV3Small': (_app.MobileNetV3Small, {'include_preprocessing': True}, None),
    'MobileNetV3Large': (_app.MobileNetV3Large, {'include_preprocessing': True}, None),
    'EfficientNetB0': (_app.EfficientNetB0, {}, None),
    'EfficientNetB1': (_app.EfficientNetB1, {}, None),
    'EfficientNetB2': (_app.EfficientNetB2, {}, None),
    'EfficientNetB3': (_app.EfficientNetB3, {}, None),
    'EfficientNetV2B0': (_app.EfficientNetV2B0, {'include_preprocessing': True}, None),
    'EfficientNetV2B1': (_app.EfficientNetV2B1, {'include_preprocessing': True}, None),
    'EfficientNetV2B2': (_app.EfficientNetV2B2, {'include_preprocessing': True}, None),
    'EfficientNetV2B3': (_app.EfficientNetV2B3, {'include_preprocessing': True}, None),
    'EfficientNetV2S': (_app.EfficientNetV2S, {'include_preprocessing': True}, None),
    'MobileNetV2': (_app.MobileNetV2, {}, _app.mobilenet_v2.preprocess_input),
    'ResNet50': (_app.ResNet50, {}, _app.resnet50.preprocess_input),
    'ConvNeXtTiny': (_app.ConvNeXtTiny, {'include_preprocessing': True}, None),
}


def make_backbone(name, size, inp):
    # 入力は常に RGB 0〜255。必要な正規化はモデルのグラフ内に入れるので、アプリ側の前処理はバックボーンに依らない
    if name not in BACKBONES:
        raise ValueError(f'未対応のバックボーン: {name}（選択肢: {list(BACKBONES)}）')
    cls, kwargs, preprocess = BACKBONES[name]
    base = cls(input_shape=(size, size, 3), include_top=False, weights='imagenet', **kwargs)
    base.trainable = False
    x = preprocess(inp) if preprocess else inp
    return base(x, training=False), base


def nearest_aspect(ratio):
    # ratio = 長辺/短辺 (>=1)
    return int(np.argmin([abs(math.log(ratio / (a / b))) for a, b in ASPECTS]))


def canvas_size(w, h, deg):
    # 回転後の画像全体が収まるキャンバスのサイズ（crop の基準）
    t = math.radians(abs(deg))
    return w * math.cos(t) + h * math.sin(t), w * math.sin(t) + h * math.cos(t)


def rotate_expand(img, deg):
    # deg > 0 で時計回りに回転し、キャンバスを拡張（はみ出し部分は黒）
    h, w = img.shape[:2]
    bw, bh = canvas_size(w, h, deg)
    BW, BH = max(1, int(round(bw))), max(1, int(round(bh)))
    M = cv2.getRotationMatrix2D((w / 2, h / 2), -deg, 1.0)
    M[0, 2] += BW / 2 - w / 2
    M[1, 2] += BH / 2 - h / 2
    return cv2.warpAffine(img, M, (BW, BH), flags=cv2.INTER_LINEAR, borderValue=(0, 0, 0))


def inscribed_scale(w, h, deg):
    # 回転しても黒い隅が出ない、同じ縦横比の中央矩形の縮尺
    t = math.radians(abs(deg))
    return min(w / (w * math.cos(t) + h * math.sin(t)), h / (w * math.sin(t) + h * math.cos(t)))


def rotate_zoom(img, deg, scale):
    # 時計回りに deg 回転し、中央の scale 倍の領域を元サイズに拡大
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), -deg, 1.0 / scale)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)


def letterbox(img, size):
    # 縦横比を保ったまま正方形に収める（角度の情報を歪ませないため）
    h, w = img.shape[:2]
    s = size / max(h, w)
    nw, nh = max(1, round(w * s)), max(1, round(h * s))
    out = np.zeros((size, size, 3), np.uint8)
    x0, y0 = (size - nw) // 2, (size - nh) // 2
    out[y0:y0 + nh, x0:x0 + nw] = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
    return out


def crop_condition(aspect_idx, portrait, canvas_w, canvas_h):
    # トリミングモデルへの条件ベクトル。最後の要素は「正規化座標での幅/高さ」の対数
    a, b = ASPECTS[aspect_idx]
    target = (b / a) if portrait else (a / b)
    v = np.zeros(COND_DIM, np.float32)
    v[aspect_idx] = 1.0
    v[NUM_ASPECTS] = float(portrait)
    v[NUM_ASPECTS + 1] = math.log(target / (canvas_w / canvas_h))
    return v


def color_jitter(img, rng):
    x = img.astype(np.float32)
    x = x * rng.uniform(0.8, 1.2) + rng.uniform(-20, 20)              # コントラスト・明るさ
    gray = x.mean(axis=2, keepdims=True)
    x = gray + (x - gray) * rng.uniform(0.7, 1.3)                       # 彩度
    return np.clip(x, 0, 255).astype(np.uint8)


def imread_rgb(path):
    buf = np.fromfile(path, np.uint8)
    img = cv2.imdecode(buf, cv2.IMREAD_COLOR)
    return None if img is None else cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

## アノテーション読み込み

In [ ]:
def group_of(stem):
    # 同じ撮影シーンが学習と検証に分かれないためのグループ名
    m = re.match(r'^(.*)_(\d+)$', stem)
    if m and not re.fullmatch(r'[A-Za-z]+', m.group(1)):
        return m.group(1)                                  # 「車種　区間_連番」
    m = re.match(r'^([A-Za-z_]*)(\d+)$', stem)
    if m:
        return f'{m.group(1)}{int(m.group(2)) // 100}'     # PA181600 などカメラ連番は100枚単位
    return stem


images_by_name = {unicodedata.normalize('NFC', os.path.basename(p)): p
                  for p in glob.glob(os.path.join(DATA_DIR, '*'))
                  if p.lower().endswith(('.jpg', '.jpeg', '.png'))}

records, skipped = [], []
for jp in sorted(glob.glob(os.path.join(DATA_DIR, '*.json'))):
    name = unicodedata.normalize('NFC', os.path.basename(jp)[:-len('.json')])
    try:
        with open(jp, encoding='utf-8-sig') as f:
            d = json.load(f)
    except Exception:
        skipped.append((name, '壊れた/空のJSON')); continue
    if name not in images_by_name:
        skipped.append((name, '画像なし')); continue
    e = d.get('edit', d)
    c = e.get('crop') or {}
    if not c or c.get('width', 0) <= 0 or c.get('height', 0) <= 0:
        skipped.append((name, 'cropなし')); continue
    records.append(dict(
        name=name, path=images_by_name[name],
        angle=float(e.get('rotation_degrees') or 0.0),
        aw=e.get('aspect_width'), ah=e.get('aspect_height'),
        portrait=bool(e.get('portrait')),
        crop=np.array([c['left'], c['top'], c['width'], c['height']], np.float32),
        group=group_of(os.path.splitext(name)[0]),
    ))

print(f'有効 {len(records)} 件 / スキップ {len(skipped)} 件')
for s in skipped[:10]:
    print('  skip:', s)

if QUICK_TEST:
    random.Random(SEED).shuffle(records)
    records = records[:160]

## 画像のキャッシュとラベルの確定
全画像が `MAX_MEMORY_CACHE_GB` に収まればメモリに、収まらなければ縮小画像をディスクに保存して学習中に読み込みます（2回目以降はディスクキャッシュを再利用）。

In [ ]:
import hashlib
from PIL import Image as PILImage

def load_resized(path):
    img = imread_rgb(path)
    if img is None:
        return None
    h, w = img.shape[:2]
    s = CACHE_LONG_SIDE / max(h, w)
    if s < 1:
        img = cv2.resize(img, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)
    return img

def header_size(path):
    with PILImage.open(path) as im:
        return im.size

def cache_file(rec):
    key = hashlib.md5(f"{rec['name']}|{os.path.getsize(rec['path'])}|{CACHE_LONG_SIDE}".encode()).hexdigest()
    return os.path.join(DISK_CACHE_DIR, key + '.jpg')

with ThreadPoolExecutor(8) as ex:
    orig_sizes = list(ex.map(lambda r: header_size(r['path']), records))
est_gb = sum(min(1.0, CACHE_LONG_SIDE / max(w, h)) ** 2 * w * h * 3 for w, h in orig_sizes) / 1e9
use_memory = (est_gb <= MAX_MEMORY_CACHE_GB) if CACHE_IN_MEMORY == 'auto' else bool(CACHE_IN_MEMORY)
print(f'{len(records)} 枚 / 推定 {est_gb:.2f} GB -> {"メモリ" if use_memory else "ディスク"}にキャッシュ')

def prepare_one(rec):
    if use_memory:
        img = load_resized(rec['path'])
        return (None, None) if img is None else (img, (img.shape[1], img.shape[0]))
    cf = cache_file(rec)
    if not os.path.exists(cf):
        img = load_resized(rec['path'])
        if img is None:
            return None, None
        cv2.imencode('.jpg', cv2.cvtColor(img, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 95])[1].tofile(cf)
    return cf, header_size(cf)

os.makedirs(DISK_CACHE_DIR, exist_ok=True)
with ThreadPoolExecutor(8) as ex:
    prepared = list(ex.map(prepare_one, records))
unreadable = [r['name'] for r, (x, _) in zip(records, prepared) if x is None]
if unreadable:
    print('読み込めない画像をスキップ:', unreadable[:10])
records = [r for r, (x, _) in zip(records, prepared) if x is not None]
prepared = [p for p in prepared if p[0] is not None]


class ImageStore:
    # IMAGES[j] で RGB 画像を返す（メモリ上 or ディスクキャッシュから読み込み）
    def __init__(self, items, in_memory):
        self.items, self.in_memory = items, in_memory

    def __len__(self):
        return len(self.items)

    def __getitem__(self, j):
        return self.items[j] if self.in_memory else imread_rgb(self.items[j])


IMAGES = ImageStore([x for x, _ in prepared], use_memory)
SIZES = [s for _, s in prepared]

for rec, (w, h) in zip(records, SIZES):
    bw, bh = canvas_size(w, h, rec['angle'])
    l, t, cw, ch = rec['crop']
    px_ratio = (cw * bw) / (ch * bh)
    if rec['aw'] is None:                    # 元の比率のまま → 実際の crop 比率から決める
        rec['aspect_idx'] = nearest_aspect(max(px_ratio, 1 / px_ratio))
        rec['portrait'] = px_ratio < 1
    else:
        rec['aspect_idx'] = ASPECTS.index((rec['aw'], rec['ah'])) if (rec['aw'], rec['ah']) in ASPECTS \
            else nearest_aspect(max(rec['aw'], rec['ah']) / min(rec['aw'], rec['ah']))
    l, t = max(0.0, l), max(0.0, t)
    rec['crop'] = np.array([l, t, min(cw, 1 - l), min(ch, 1 - t)], np.float32)

ANGLES = np.array([r['angle'] for r in records], np.float32)
ASPECT_IDX = np.array([r['aspect_idx'] for r in records], np.int32)
PORTRAIT = np.array([r['portrait'] for r in records], np.float32)
CROPS = np.stack([r['crop'] for r in records])

print('回転角: min %.2f / max %.2f / 平均 %.2f / 標準偏差 %.2f' % (ANGLES.min(), ANGLES.max(), ANGLES.mean(), ANGLES.std()))
for i, n in enumerate(ASPECT_NAMES):
    print(f'  {n:>5}: {np.sum(ASPECT_IDX == i)} 件')
print('縦構図:', int(PORTRAIT.sum()), '件')

## 学習/検証の分割（撮影シーン単位）
グループ名のハッシュで決めるので、データを追加しても既存の画像の割り当ては変わりません（前回のモデルと同じ検証データで比較できます）。

In [ ]:
def is_val_group(g):
    h = int(hashlib.md5(f'{SEED}|{g}'.encode()).hexdigest(), 16)
    return (h % 10000) < VAL_RATIO * 10000

groups = sorted({r['group'] for r in records})
val_groups = {g for g in groups if is_val_group(g)}

VAL_IDX = np.array([i for i, r in enumerate(records) if r['group'] in val_groups])
TRAIN_IDX = np.array([i for i, r in enumerate(records) if r['group'] not in val_groups])
print(f'train {len(TRAIN_IDX)} / val {len(VAL_IDX)}  (グループ数 {len(groups)})')

## 教師データの可視化（回転・crop の解釈が正しいか確認）
緑枠が教師データの crop です。回転後の画像上で意図どおりの位置に来ていれば OK。

In [ ]:
def draw_box(ax, box, color, label=None):
    l, t, w, h = box
    ax.add_patch(plt.Rectangle((l, t), w, h, fill=False, ec=color, lw=2, label=label))

def show_on_canvas(ax, img, deg):
    canvas = rotate_expand(img, deg)
    ax.imshow(canvas, extent=(0, 1, 1, 0)); ax.set_xlim(0, 1); ax.set_ylim(1, 0); ax.axis('off')

sample = sorted(range(len(records)), key=lambda i: -abs(ANGLES[i]))[:4] + random.Random(0).sample(range(len(records)), 4)
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, i in zip(axes.ravel(), sample):
    show_on_canvas(ax, IMAGES[i], ANGLES[i])
    draw_box(ax, CROPS[i], 'lime')
    ax.set_title(f'#{i} rot={ANGLES[i]:.2f}  {ASPECT_NAMES[ASPECT_IDX[i]]}{" P" if PORTRAIT[i] else ""}')
plt.tight_layout(); plt.show()

# 1. 解析モデル（回転角 + 縦横比）

- 入力: 元画像をレターボックスで `A_SIZE×A_SIZE` に（RGB, 0〜255 の float32。正規化はモデル内）。
- 水増し: 左右反転（角度の符号も反転）、ランダム回転（加えた分だけ正解角度を補正）、色調。
- 縦横比はクラスの偏りが大きいのでクラス重み付きの交差エントロピーを使います。

In [ ]:
class AnalyzerData(keras.utils.PyDataset):
    def __init__(self, idx, batch_size, augment, **kwargs):
        super().__init__(**kwargs)
        self.idx, self.bs, self.augment = np.array(idx), batch_size, augment
        self.order = self.idx.copy()
        self.on_epoch_end()

    def __len__(self):
        return math.ceil(len(self.idx) / self.bs)

    def on_epoch_end(self):
        self.order = np.random.permutation(self.idx) if self.augment else self.idx.copy()

    def __getitem__(self, b):
        ids = self.order[b * self.bs:(b + 1) * self.bs]
        rng = np.random.default_rng()
        X = np.zeros((len(ids), A_SIZE, A_SIZE, 3), np.float32)
        rot = np.zeros((len(ids), 1), np.float32)
        for k, j in enumerate(ids):
            img, ang = IMAGES[j], float(ANGLES[j])
            if self.augment:
                if rng.random() < 0.5:
                    img, ang = img[:, ::-1], -ang
                if rng.random() < ROT_AUG_PROB:
                    phi = rng.uniform(-ROT_AUG_MAX, ROT_AUG_MAX)
                    h, w = img.shape[:2]
                    img = rotate_zoom(np.ascontiguousarray(img), phi, inscribed_scale(w, h, phi) * rng.uniform(0.9, 1.0))
                    ang -= phi                 # 既に phi だけ時計回りに回したので残りは ang - phi
                img = color_jitter(img, rng)
            X[k] = letterbox(np.ascontiguousarray(img), A_SIZE)
            rot[k, 0] = ang
        y = {'rotation': rot,
             'aspect': ASPECT_IDX[ids].astype(np.int32),
             'portrait': PORTRAIT[ids][:, None]}
        return X, y


def weighted_sparse_ce(class_weights):
    w = np.asarray(class_weights, np.float32)
    def loss(y_true, y_pred):
        y_true = ops.cast(ops.reshape(y_true, (-1,)), 'int32')
        ce = keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
        return ce * ops.take(ops.convert_to_tensor(w), y_true)
    return loss


def weighted_bce(pos_weight):
    def loss(y_true, y_pred):
        y_true = ops.cast(y_true, 'float32')
        p = ops.clip(y_pred, 1e-7, 1 - 1e-7)
        return -ops.mean(pos_weight * y_true * ops.log(p) + (1 - y_true) * ops.log(1 - p), axis=-1)
    return loss


def freeze_bn(model):
    for l in model.layers:
        if isinstance(l, layers.BatchNormalization):
            l.trainable = False


def build_analyzer():
    inp = keras.Input((A_SIZE, A_SIZE, 3), name='image')
    x, base = make_backbone(A_BACKBONE, A_SIZE, inp)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    r = layers.Dense(256, activation='relu')(x)
    rotation = layers.Dense(1, name='rotation')(r)
    aspect = layers.Dense(NUM_ASPECTS, activation='softmax', name='aspect')(x)
    portrait = layers.Dense(1, activation='sigmoid', name='portrait')(x)
    model = keras.Model(inp, {'rotation': rotation, 'aspect': aspect, 'portrait': portrait}, name='trim_analyzer')
    return model, base


counts = np.bincount(ASPECT_IDX[TRAIN_IDX], minlength=NUM_ASPECTS).astype(np.float32)
aspect_w = (counts.sum() / np.maximum(counts, 1) / NUM_ASPECTS) ** 0.5     # 偏りを緩やかに補正
pos = PORTRAIT[TRAIN_IDX].mean()
portrait_pw = float(np.clip((1 - pos) / max(pos, 1e-3), 1, 20))
print('aspect class weights:', dict(zip(ASPECT_NAMES, aspect_w.round(2))), ' portrait pos_weight:', round(portrait_pw, 1))

def compile_analyzer(model, lr):
    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss={'rotation': keras.losses.Huber(delta=1.0),
              'aspect': weighted_sparse_ce(aspect_w),
              'portrait': weighted_bce(portrait_pw)},
        loss_weights={'rotation': 1.0, 'aspect': 1.0, 'portrait': 0.3},
        metrics={'rotation': [keras.metrics.MeanAbsoluteError(name='mae')],
                 'aspect': [keras.metrics.SparseCategoricalAccuracy(name='acc')],
                 'portrait': [keras.metrics.BinaryAccuracy(name='acc')]},
    )

analyzer, a_base = build_analyzer()
print(f'解析モデル: {A_BACKBONE}, 入力 {A_SIZE}px, パラメータ数 {analyzer.count_params():,}')
a_train = AnalyzerData(TRAIN_IDX, A_BATCH, augment=True, workers=DATA_WORKERS)
a_val = AnalyzerData(VAL_IDX, A_BATCH, augment=False, workers=DATA_WORKERS)

In [ ]:
A_CKPT = os.path.join(RUN_DIR, 'trim_analyzer.keras')

def callbacks(ckpt):
    return [keras.callbacks.ModelCheckpoint(ckpt, monitor='val_loss', save_best_only=True),
            keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6),
            keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)]

def init_weights(model, filename):
    # 前回の学習結果から重みを引き継ぐ。引き継いだ場合はバックボーン凍結の段階を省略する
    if not INIT_FROM_DIR:
        return False
    model.load_weights(os.path.join(INIT_FROM_DIR, filename))
    print('重みを引き継ぎ:', os.path.join(INIT_FROM_DIR, filename))
    return True

def run_fit(model, train, val, epochs, ckpt):
    return model.fit(train, validation_data=val, epochs=epochs, callbacks=callbacks(ckpt)) if epochs > 0 else None

resumed = init_weights(analyzer, 'trim_analyzer.keras')

# ステージ1: バックボーン凍結
compile_analyzer(analyzer, 1e-3)
hist_a1 = run_fit(analyzer, a_train, a_val, 0 if resumed else A_EPOCHS_HEAD, A_CKPT)

# ステージ2: 全体を微調整（BatchNorm は固定）
a_base.trainable = True
freeze_bn(a_base)
compile_analyzer(analyzer, 5e-5 if resumed else 1e-4)
hist_a2 = run_fit(analyzer, a_train, a_val, A_EPOCHS_FINE, A_CKPT)
analyzer.save(A_CKPT)

# 2. トリミングモデル（指定比率での BBox）

- 入力1 `image`: 回転後キャンバス（`rotate_expand` の結果）を `B_SIZE×B_SIZE` に単純リサイズ（座標が 0〜1 のまま対応するため）。
- 入力2 `condition`: `crop_condition(aspect_idx, portrait, キャンバス幅, キャンバス高さ)` の 7 次元ベクトル。
- 出力 `box`: `[left, top, width, height]`。モデル内部で「中心・面積」から指定比率どおりの矩形を作り、キャンバスからはみ出さないよう補正するので、**出力は必ず指定比率になります**。
- 損失: L1 + GIoU。

In [ ]:
@keras.saving.register_keras_serializable(package='autotrim')
class AspectBox(layers.Layer):
    # (cx, cy, 面積) の生出力と log(幅/高さ) から、比率を厳密に守る [left, top, width, height] を作る
    def call(self, params, cond):
        cx = ops.sigmoid(params[:, 0:1])
        cy = ops.sigmoid(params[:, 1:2])
        area = ops.sigmoid(params[:, 2:3]) * 0.98 + 0.02
        r = ops.exp(cond[:, NUM_ASPECTS + 1:NUM_ASPECTS + 2])
        w = ops.sqrt(area * r)
        h = ops.sqrt(area / r)
        k = ops.minimum(1.0, ops.minimum(1.0 / w, 1.0 / h))
        w, h = w * k, h * k
        cx = ops.maximum(ops.minimum(cx, 1.0 - w / 2), w / 2)
        cy = ops.maximum(ops.minimum(cy, 1.0 - h / 2), h / 2)
        return ops.concatenate([cx - w / 2, cy - h / 2, w, h], axis=-1)


def build_cropper():
    img = keras.Input((B_SIZE, B_SIZE, 3), name='image')
    cond = keras.Input((COND_DIM,), name='condition')
    f, base = make_backbone(B_BACKBONE, B_SIZE, img)
    f = layers.Conv2D(64, 1, activation='relu')(f)        # 位置情報を残すため GAP せずに平坦化
    f = layers.Flatten()(f)
    f = layers.Dropout(0.3)(f)
    c = layers.Dense(64, activation='relu')(cond)
    x = layers.Concatenate()([f, c])
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Concatenate()([x, c])
    x = layers.Dense(128, activation='relu')(x)
    params = layers.Dense(3, bias_initializer=keras.initializers.Constant([0.0, 0.0, 0.85]))(x)
    box = AspectBox(name='box')(params, cond)
    return keras.Model([img, cond], box, name='trim_cropper'), base


def box_iou_terms(y_true, y_pred):
    tl, tt, tw, th = [y_true[:, i] for i in range(4)]
    pl, pt, pw, ph = [y_pred[:, i] for i in range(4)]
    ix = ops.maximum(0.0, ops.minimum(tl + tw, pl + pw) - ops.maximum(tl, pl))
    iy = ops.maximum(0.0, ops.minimum(tt + th, pt + ph) - ops.maximum(tt, pt))
    inter = ix * iy
    union = tw * th + pw * ph - inter + 1e-7
    iou = inter / union
    ex = ops.maximum(tl + tw, pl + pw) - ops.minimum(tl, pl)
    ey = ops.maximum(tt + th, pt + ph) - ops.minimum(tt, pt)
    enclose = ex * ey + 1e-7
    giou = iou - (enclose - union) / enclose
    return iou, giou


@keras.saving.register_keras_serializable(package='autotrim')
def box_loss(y_true, y_pred):
    _, giou = box_iou_terms(y_true, y_pred)
    return 2.0 * ops.mean(ops.abs(y_true - y_pred), axis=-1) + (1.0 - giou)


@keras.saving.register_keras_serializable(package='autotrim')
def box_iou(y_true, y_pred):
    iou, _ = box_iou_terms(y_true, y_pred)
    return iou


def make_crop_input(img, deg, aspect_idx, portrait):
    canvas = rotate_expand(img, deg)
    cond = crop_condition(aspect_idx, portrait, canvas.shape[1], canvas.shape[0])
    x = cv2.resize(canvas, (B_SIZE, B_SIZE), interpolation=cv2.INTER_AREA).astype(np.float32)
    return x, cond, canvas


class CropperData(keras.utils.PyDataset):
    def __init__(self, idx, batch_size, augment, **kwargs):
        super().__init__(**kwargs)
        self.idx, self.bs, self.augment = np.array(idx), batch_size, augment
        self.order = self.idx.copy()
        self.on_epoch_end()

    def __len__(self):
        return math.ceil(len(self.idx) / self.bs)

    def on_epoch_end(self):
        self.order = np.random.permutation(self.idx) if self.augment else self.idx.copy()

    def __getitem__(self, b):
        ids = self.order[b * self.bs:(b + 1) * self.bs]
        rng = np.random.default_rng()
        X = np.zeros((len(ids), B_SIZE, B_SIZE, 3), np.float32)
        C = np.zeros((len(ids), COND_DIM), np.float32)
        Y = np.zeros((len(ids), 4), np.float32)
        for k, j in enumerate(ids):
            canvas = rotate_expand(IMAGES[j], float(ANGLES[j]))
            l, t, w, h = CROPS[j].astype(np.float64)
            if self.augment:
                if rng.random() < 0.5:
                    canvas = canvas[:, ::-1]
                    l = 1 - l - w
                l, t = max(l, 0.0), max(t, 0.0)
                w, h = min(w, 1.0 - l), min(h, 1.0 - t)
                if rng.random() < CROP_AUG_PROB:
                    H, W = canvas.shape[:2]
                    x0 = int(rng.uniform(0, l) * W); x1 = int(math.ceil(rng.uniform(l + w, 1) * W))
                    y0 = int(rng.uniform(0, t) * H); y1 = int(math.ceil(rng.uniform(t + h, 1) * H))
                    x1, y1 = max(x1, x0 + 2), max(y1, y0 + 2)
                    canvas = canvas[y0:y1, x0:x1]
                    nW, nH = x1 - x0, y1 - y0
                    l, t, w, h = (l * W - x0) / nW, (t * H - y0) / nH, w * W / nW, h * H / nH
                canvas = color_jitter(canvas, rng)
            canvas = np.ascontiguousarray(canvas)
            C[k] = crop_condition(int(ASPECT_IDX[j]), bool(PORTRAIT[j]), canvas.shape[1], canvas.shape[0])
            X[k] = cv2.resize(canvas, (B_SIZE, B_SIZE), interpolation=cv2.INTER_AREA)
            Y[k] = np.clip([l, t, w, h], 0, 1)
        return (X, C), Y


def compile_cropper(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss=box_loss, metrics=[box_iou])

cropper, b_base = build_cropper()
print(f'トリミングモデル: {B_BACKBONE}, 入力 {B_SIZE}px, パラメータ数 {cropper.count_params():,}')
b_train = CropperData(TRAIN_IDX, B_BATCH, augment=True, workers=DATA_WORKERS)
b_val = CropperData(VAL_IDX, B_BATCH, augment=False, workers=DATA_WORKERS)

In [ ]:
B_CKPT = os.path.join(RUN_DIR, 'trim_cropper.keras')
resumed = init_weights(cropper, 'trim_cropper.keras')

compile_cropper(cropper, 1e-3)
hist_b1 = run_fit(cropper, b_train, b_val, 0 if resumed else B_EPOCHS_HEAD, B_CKPT)

b_base.trainable = True
freeze_bn(b_base)
compile_cropper(cropper, 5e-5 if resumed else 1e-4)
hist_b2 = run_fit(cropper, b_train, b_val, B_EPOCHS_FINE, B_CKPT)
cropper.save(B_CKPT)

## 学習曲線

In [ ]:
def plot_hist(hists, keys, title):
    fig, axes = plt.subplots(1, len(keys), figsize=(5 * len(keys), 3.5))
    for ax, k in zip(np.atleast_1d(axes), keys):
        for split in ['', 'val_']:
            vals = sum([h.history.get(split + k, []) for h in hists if h is not None], [])
            ax.plot(vals, label=split + k)
        ax.set_title(f'{title}: {k}'); ax.legend()
    plt.tight_layout(); plt.show()

plot_hist([hist_a1, hist_a2], ['loss', 'rotation_mae', 'aspect_acc'], 'analyzer')
plot_hist([hist_b1, hist_b2], ['loss', 'box_iou'], 'cropper')

# 3. 検証データでの評価
ベースライン（回転0度／最多クラス／中央に最大の矩形）と比べて改善しているかを確認します。

In [ ]:
def center_box(cond):
    r = math.exp(cond[NUM_ASPECTS + 1])
    w, h = min(1.0, r), min(1.0, 1.0 / r)
    return np.array([(1 - w) / 2, (1 - h) / 2, w, h], np.float32)

pa = analyzer.predict(a_val, verbose=0)
rot_pred = pa['rotation'][:, 0]
asp_pred = pa['aspect'].argmax(-1)
por_pred = pa['portrait'][:, 0] > 0.5
gt = ANGLES[VAL_IDX]
print(f'回転角 MAE: {np.abs(rot_pred - gt).mean():.3f} 度   (常に0度の場合: {np.abs(gt).mean():.3f} 度)')
print(f'縦横比 正解率: {(asp_pred == ASPECT_IDX[VAL_IDX]).mean():.3f}   (最多クラスの場合: {(ASPECT_IDX[VAL_IDX] == np.bincount(ASPECT_IDX[TRAIN_IDX]).argmax()).mean():.3f})')
print(f'縦構図 正解率: {(por_pred == (PORTRAIT[VAL_IDX] > 0.5)).mean():.3f}')

cm = np.zeros((NUM_ASPECTS, NUM_ASPECTS), int)
for t_, p_ in zip(ASPECT_IDX[VAL_IDX], asp_pred):
    cm[t_, p_] += 1
print('混同行列（行=正解, 列=予測）', ASPECT_NAMES); print(cm)

box_pred = cropper.predict(b_val, verbose=0)
Yall = np.concatenate([b_val[i][1] for i in range(len(b_val))])
Call = np.concatenate([b_val[i][0][1] for i in range(len(b_val))])
iou_model = keras.ops.convert_to_numpy(box_iou(Yall, box_pred)).mean()
iou_center = keras.ops.convert_to_numpy(box_iou(Yall, np.stack([center_box(c) for c in Call]))).mean()
print(f'BBox 平均IoU: {iou_model:.3f}   (中央に最大矩形の場合: {iou_center:.3f})')

metrics = {
    'run_name': RUN_NAME, 'num_train': int(len(TRAIN_IDX)), 'num_val': int(len(VAL_IDX)),
    'rotation_mae': float(np.abs(rot_pred - gt).mean()),
    'aspect_acc': float((asp_pred == ASPECT_IDX[VAL_IDX]).mean()),
    'portrait_acc': float((por_pred == (PORTRAIT[VAL_IDX] > 0.5)).mean()),
    'box_iou': float(iou_model),
}
with open(os.path.join(RUN_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

# これまでの学習結果を並べて比較（データを増やしたときの効果確認用）
history_rows = []
for p in sorted(glob.glob(os.path.join(OUTPUT_DIR, '*', 'metrics.json'))):
    with open(p) as f:
        m = json.load(f)
    with open(os.path.join(os.path.dirname(p), 'config.json')) as f:
        cfg = json.load(f)
    history_rows.append((m['run_name'], cfg['A_BACKBONE'], cfg['B_BACKBONE'], m['num_train'], m['num_val'],
                         m['rotation_mae'], m['aspect_acc'], m['box_iou']))
print(f'\n{"run":<14}{"analyzer":<18}{"cropper":<18}{"train":>6}{"val":>6}{"rotMAE":>8}{"aspAcc":>8}{"IoU":>7}')
for r in history_rows:
    print(f'{r[0]:<14}{r[1]:<18}{r[2]:<18}{r[3]:>6}{r[4]:>6}{r[5]:>8.3f}{r[6]:>8.3f}{r[7]:>7.3f}')

## 予測結果の可視化
緑 = 教師データ、赤 = モデル予測（比率は教師データの比率を指定）。

In [ ]:
show = random.Random(1).sample(list(range(len(VAL_IDX))), min(8, len(VAL_IDX)))
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, k in zip(axes.ravel(), show):
    j = VAL_IDX[k]
    show_on_canvas(ax, IMAGES[j], ANGLES[j])
    draw_box(ax, CROPS[j], 'lime'); draw_box(ax, box_pred[k], 'red')
    ax.set_title(f'rot gt={ANGLES[j]:.2f} pred={rot_pred[k]:.2f} | {ASPECT_NAMES[ASPECT_IDX[j]]}->{ASPECT_NAMES[asp_pred[k]]}')
plt.tight_layout(); plt.show()

# 4. 推論パイプライン（アプリ側で再現する処理）

`auto_trim()` は教師データの `edit` と同じ形式の dict を返します。`aspect` を指定するとその比率で BBox を出します（`None` ならモデルの推定比率）。

In [ ]:
analyzer_inf = keras.models.load_model(A_CKPT, compile=False)
cropper_inf = keras.models.load_model(B_CKPT, compile=False)

def auto_trim(img_rgb, aspect=None, portrait=None):
    # img_rgb: HxWx3 uint8 (RGB)。aspect: '3:2' などの文字列、または None
    out = analyzer_inf.predict(letterbox(img_rgb, A_SIZE)[None].astype(np.float32), verbose=0)
    deg = float(np.clip(out['rotation'][0, 0], -45, 45))
    aspect_idx = ASPECT_NAMES.index(aspect) if aspect else int(out['aspect'][0].argmax())
    if portrait is None:
        portrait = bool(out['portrait'][0, 0] > 0.5)
    x, cond, canvas = make_crop_input(img_rgb, deg, aspect_idx, portrait)
    l, t, w, h = cropper_inf.predict([x[None], cond[None]], verbose=0)[0].tolist()
    aw, ah = ASPECTS[aspect_idx]
    return {
        'rotation_degrees': deg,
        'aspect_width': aw, 'aspect_height': ah, 'portrait': portrait,
        'crop': {'left': l, 'top': t, 'width': w, 'height': h},
        'aspect_probs': dict(zip(ASPECT_NAMES, out['aspect'][0].round(3).tolist())),
    }

def apply_edit(img_rgb, edit):
    canvas = rotate_expand(img_rgb, edit['rotation_degrees'])
    H, W = canvas.shape[:2]; c = edit['crop']
    return canvas[int(c['top'] * H):int((c['top'] + c['height']) * H), int(c['left'] * W):int((c['left'] + c['width']) * W)]

j = VAL_IDX[0]
img = imread_rgb(records[j]['path'])
edit = auto_trim(img)
print(json.dumps(edit, indent=2, ensure_ascii=False))
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(img); axes[0].set_title('input'); axes[0].axis('off')
axes[1].imshow(apply_edit(img, edit)); axes[1].set_title(f'auto ({ASPECT_NAMES[ASPECTS.index((edit["aspect_width"], edit["aspect_height"]))]})'); axes[1].axis('off')
edit169 = auto_trim(img, aspect='16:9', portrait=False)
axes[2].imshow(apply_edit(img, edit169)); axes[2].set_title('aspect=16:9 (specified)'); axes[2].axis('off')
plt.tight_layout(); plt.show()

# 5. 保存（TFLite 変換用）

`.keras` は学習中に `OUTPUT_DIR/RUN_NAME` へ保存済みです（設定は `config.json`、評価結果は `metrics.json`）。下のセルで TFLite 変換に使いやすい SavedModel も書き出します。
バックボーンを変えても入力は常に RGB 0〜255 で、正規化はモデル内で行うため、アプリ側の前処理は変わりません（変わるのは `A_SIZE` / `B_SIZE` を変えた場合の入力サイズだけです）。
事前に用意してある TFLite 変換処理には、`.keras`（`keras.models.load_model(..., compile=False)` で読み込み）か、この SavedModel を渡してください。
`AspectBox` はカスタムレイヤーなので、別のノートブックで `.keras` を読み込む場合は上の `AspectBox` クラス定義セルを先に実行してください（SavedModel なら不要です）。

### アプリ側の入出力仕様
| モデル | 入力 | 出力 |
|---|---|---|
| `trim_analyzer` | `image`: `[1, A_SIZE, A_SIZE, 3]` float32, RGB 0〜255, 縦横比を保って縮小し中央に配置、余白は黒 | `rotation` `[1,1]`（度, 時計回りが正, -45〜45 にクリップ）, `aspect` `[1,5]`（`1:1, 5:4, 4:3, 3:2, 16:9` の確率）, `portrait` `[1,1]` |
| `trim_cropper` | `image`: `[1, B_SIZE, B_SIZE, 3]` float32, 回転後キャンバスを縦横比を無視してリサイズ / `condition`: `[1,7]` = 比率 one-hot(5) + portrait(0/1) + `log(目標の幅/高さ ÷ キャンバスの幅/高さ)` | `box` `[1,4]` = `left, top, width, height`（キャンバス基準の 0〜1、JSON の `crop` と同じ） |

回転後キャンバスのサイズは `W·cosθ + H·sinθ` × `W·sinθ + H·cosθ`（θ は角度の絶対値）です。

In [ ]:
analyzer_inf.export(os.path.join(RUN_DIR, 'trim_analyzer_savedmodel'))
cropper_inf.export(os.path.join(RUN_DIR, 'trim_cropper_savedmodel'))
print(os.listdir(RUN_DIR))